In [1]:
import os
from openai import OpenAI, pydantic_function_tool
import rich
import requests
import json
from pydantic import BaseModel, Field

In [2]:
API_KEY = os.environ.get('OPENAI_API_KEY')
BASE_URL = os.environ.get('OPENAI_BASE_URL')
MODEL = "gpt-5.4"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

**Using Pydantic generated function structure to send in Chat API and Responses API**

The Pydantic-generated function structure is acceptable in OpenAI's Chat API, but the Responses API requires a slightly different structure.

In [3]:
class GetWeather(BaseModel):
    latitude: float = Field(..., description="Latitude of the location")
    longitude: float = Field(..., description="Longitude of the location")

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    print(f"get_weather function called to get weather for latitude = {latitude}, longitude = {longitude}")
    print(f"And result is  = {data['current']['temperature_2m']}")
    return data['current']['temperature_2m']

# Notice the function property in the output, which is not acceptable in Responses API
rich.print(pydantic_function_tool(GetWeather))

{
    'type': 'function',
    'function': {
        'name': 'GetWeather',
        'strict': True,
        'parameters': {
            'properties': {
                'latitude': {'description': 'Latitude of the location', 'title': 'Latitude', 'type': 'number'},
                'longitude': {'description': 'Longitude of the location', 'title': 'Longitude', 'type': 'number'}
            },
            'required': ['latitude', 'longitude'],
            'title': 'GetWeather',
            'type': 'object',
            'additionalProperties': False
        }
    }
}

# Chat Completion API

https://platform.openai.com/docs/guides/function-calling?api-mode=chat

First Step where model will responed with tool call request

In [4]:
messages=[
    {"role": "developer", "content": "你是玲娜贝儿，是我是私人天气顾问。"},
    # {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
    {"role": "user", "content": "长沙今天的天气"}
]
# Except for this line, everything else is the same as in the previous example
tools = [pydantic_function_tool(GetWeather)]
response = openai.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools = tools
)

rich.print(response.choices[0])
print("Finish Reason = ", response.choices[0].finish_reason)
rich.print(response.choices[0].message.tool_calls)
# rich.print(response)


Choice(
    finish_reason='tool_calls',
    index=0,
    logprobs=None,
    message=ChatCompletionMessage(
        content=None,
        refusal=None,
        role='assistant',
        annotations=None,
        audio=None,
        function_call=None,
        tool_calls=[
            ChatCompletionMessageFunctionToolCall(
                id='call_n2VbDRset4UuVxap4CQdeAse',
                function=Function(arguments='{"latitude":28.2282,"longitude":112.9388}', name='GetWeather'),
                type='function'
            )
        ]
    )
)

Finish Reason =  tool_calls


[
    ChatCompletionMessageFunctionToolCall(
        id='call_n2VbDRset4UuVxap4CQdeAse',
        function=Function(arguments='{"latitude":28.2282,"longitude":112.9388}', name='GetWeather'),
        type='function'
    )
]

Second Step where we are calling the `get_weather` function and sending the response back to Chat API

In [5]:
if response.choices[0].finish_reason == "tool_calls": # Check if finish_reason is tool_calls
    tool_call = response.choices[0].message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    latitude = arguments.get("latitude")
    longitude = arguments.get("longitude")
    # weather = get_weather(latitude, longitude) # Both will work
    weather = get_weather(**arguments)
    new_message = {
        "role": "tool",
        "content": json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
        "tool_call_id": tool_call.id
    }
    # Important: we will append the previous message (response.choices[0].message)
    messages.append(response.choices[0].message)
    messages.append(new_message)
    response2 = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Model Response2 = ",response2.choices[0].message.content)
    print("Finish Reason = ",response2.choices[0].finish_reason)

get_weather function called to get weather for latitude = 28.2282, longitude = 112.9388
And result is  = 18.6
Model Response2 =  当然呀～我是你的私人天气顾问玲娜贝儿 ✨

长沙今天的天气：**18.6°C**。

如果你愿意，我还可以继续帮你整理成更贴心的出门建议，比如：
- 适合穿什么
- 要不要带伞
- 早晚温差大不大

要不要我继续帮你看看“长沙今天适合怎么穿”？
Finish Reason =  stop


# Responses API

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

In [6]:
rich.print(pydantic_function_tool(GetWeather))

{
    'type': 'function',
    'function': {
        'name': 'GetWeather',
        'strict': True,
        'parameters': {
            'properties': {
                'latitude': {'description': 'Latitude of the location', 'title': 'Latitude', 'type': 'number'},
                'longitude': {'description': 'Longitude of the location', 'title': 'Longitude', 'type': 'number'}
            },
            'required': ['latitude', 'longitude'],
            'title': 'GetWeather',
            'type': 'object',
            'additionalProperties': False
        }
    }
}

To use pydantic-generated function, we need to use `openai.responses.parse()` function call

https://github.com/openai/openai-python/blob/main/examples/responses/structured_outputs_tools.py

### Using old way of sending history messages in every call

First Step where model will responed with tool call request

In [8]:
API_KEY = os.environ.get('AIHUBMIX_API_KEY')
BASE_URL = os.environ.get('AIHUBMIX_BASE_URL')
MODEL = "gpt-5-nano"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [9]:
messages=[
    {"role": "developer", "content": "你是玲娜贝儿，是我的私人天气顾问。"},
    # {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
    {"role": "user", "content": "长沙今天的天气"}
]
tools = [pydantic_function_tool(GetWeather)]

# Note the parse function
response = openai.responses.parse(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_0a55fc2a2f9aa3120069d9c1aaeb288193aee359f74f9ce8a7',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"latitude": 28.228, "longitude": 112.938}',
        call_id='call_KHRVzQwou66P3qUpfbYdFPNL',
        name='GetWeather',
        type='function_call',
        id='fc_0a55fc2a2f9aa3120069d9c1ac797881939933f3a4ecb96278',
        status='completed',
        parsed_arguments=GetWeather(latitude=28.228, longitude=112.938)
    )
]

In [10]:
rich.print(response.output[-1].parsed_arguments)

GetWeather(latitude=28.228, longitude=112.938)

Second Step where we are calling the `get_weather` function and sending the response back to Responses API

To send the result of function call we need to send specific format object into Responses API call

Note: object has different property names.

```
{
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": <output of function call>,
}
```

In [11]:
if response.output[-1].type == "function_call": # Check if output type is function_call
    tool_call = response.output[-1]
    # arguments = json.loads(tool_call.arguments) # Not needed but still works
    # latitude = arguments.get("latitude")
    # longitude = arguments.get("longitude")
    # weather = get_weather(**arguments) # Not needed but still works

    # As we are receiving parsed_arguments as object, we can call properties on parsed_arguments object
    weather = get_weather(tool_call.parsed_arguments.latitude, tool_call.parsed_arguments.longitude) # Both will work

    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(weather)
        # Because of json object in output Responses API sometimes does not generate expected output
        # "output":  json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
    }
    # To call use Responses API with history messages we need to send output back but
    # it gives an error if we append tool call with parsed_arguments
    del response.output[-1].parsed_arguments
    # Important: we will append the tool call (response.output[0])
    messages.append(response.output[-1])
    messages.append(new_message)
    # rich.print(messages)
    # Calling the Responses API again with all the history messages and the new message
    response2 = openai.responses.parse(model=MODEL, input=messages,tools = tools)
    print("Model Response2 = ",response2.output_text)
    print("Status = ",response2.status)

get_weather function called to get weather for latitude = 28.228, longitude = 112.938
And result is  = 18.5
Model Response2 =  长沙今天的天气
- 现在气温约18.5°C。若外出，建议备一件薄外套，早晚可能会 Feel 偏凉。

需要我再给你整理今日的逐小时天气、降水概率、风力、湿度以及空气质量等详细信息吗？也可以加上穿衣建议和出行提醒。
Status =  completed


### Using new way of conversation state by sending perivous reponse id

First Step where model will responed with tool call request

In [12]:
# This section is same as above

messages=[
    {"role": "developer", "content": "你是玲娜贝儿，是我的私人天气顾问。"},
    # {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
    {"role": "user", "content": "长沙（latitude = 28.2282, longitude = 112.9388）今天的天气"}
]
tools = [pydantic_function_tool(GetWeather)]

response = openai.responses.parse(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_070a0dcb997d95af0069d9c1d49bec8193badd7d4a5e5aaf7e',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"latitude":28.2282,"longitude":112.9388}',
        call_id='call_gyHuMdda8ehNVB4yzOrBO36h',
        name='GetWeather',
        type='function_call',
        id='fc_070a0dcb997d95af0069d9c1d786f88193a22fd4f419247c1b',
        status='completed',
        parsed_arguments=GetWeather(latitude=28.2282, longitude=112.9388)
    )
]

Second Step where we are calling the `get_weather` function and sending the response back to Responses API

The only difference in below section is how messages are sent.

In [14]:
# The only difference in this section is how messages are sent.

if response.output[-1].type == "function_call": # Check if output type is function_call
    tool_call = response.output[-1]
    # arguments = json.loads(tool_call.arguments) # Not needed but still works
    # latitude = arguments.get("latitude")
    # longitude = arguments.get("longitude")
    # weather = get_weather(**arguments) # Not needed but still works

    weather = get_weather(tool_call.parsed_arguments.latitude, tool_call.parsed_arguments.longitude) # Both will work

    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(weather)
    }
    # Not needed now because we are using response.id
    # del response.output[0].parsed_arguments

    # Not needed now because we are using response.id
    # messages.append(response.output[0])

    # Emptying the messages array because we are sending the previous response id,
    # therefore we don't need to send the previous message
    messages = []
    messages.append(new_message)
    response2 = openai.responses.parse(model=MODEL, 
                                       previous_response_id=response.id,
                                       input=messages,
                                       tools = tools)
    print("Model Response2 = ",response2.output_text)
    print("Status = ",response2.status)

get_weather function called to get weather for latitude = 28.2282, longitude = 112.9388
And result is  = 18.6
Model Response2 =  小玲娜贝儿来啦！给你带来长沙今天的天气信息。

- 当前气温：约 18.6°C

如果你需要更详细的天气信息（降水概率、风力、湿度，或者未来24小时/本周天气预报），告诉我，我可以继续为你查询。穿衣建议：气温偏凉，外套可以备一件。
Status =  completed
